# 01 — Data Cleaning
**Goal:** Load raw simulated data, clean it, and output structured tables.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Loading raw data...")
customers = pd.read_csv('../raw_customers.csv')
products  = pd.read_csv('../raw_products.csv')
transactions = pd.read_csv('../raw_transactions.csv')

print(f"Customers: {len(customers)} rows | Transactions: {len(transactions)} rows")


## Step 1: Inspect Nulls

In [ ]:
print("=== NULL COUNTS ===")
print("Customers:\n", customers.isnull().sum())
print("\nTransactions:\n", transactions.isnull().sum())


## Step 2: Fix Dates

In [ ]:
customers['signup_date']      = pd.to_datetime(customers['signup_date'], errors='coerce')
transactions['invoice_date']  = pd.to_datetime(transactions['invoice_date'], errors='coerce')

print("Date range (transactions):", transactions['invoice_date'].min(), "→", transactions['invoice_date'].max())


## Step 3: Handle Nulls

In [ ]:
# Customers: fill city, drop truly broken rows
customers['city'] = customers['city'].fillna('Unknown')
customers = customers.dropna(subset=['customer_id'])

# Transactions: drop rows with null dates or amounts
before = len(transactions)
transactions = transactions.dropna(subset=['invoice_date', 'amount'])
print(f"Dropped {before - len(transactions)} null rows from transactions")


## Step 4: Remove Duplicates

In [ ]:
before = len(transactions)
transactions = transactions.drop_duplicates(subset=['invoice_id'])
print(f"Removed {before - len(transactions)} duplicate invoices")
print(f"Clean transactions: {len(transactions)}")


## Step 5: Validate & Recalculate

In [ ]:
transactions = transactions[transactions['amount'] > 0]
transactions = transactions[transactions['quantity'] > 0]
transactions['amount'] = transactions['quantity'] * transactions['unit_price']

print("Amount stats:\n", transactions['amount'].describe())


## Step 6: Build & Save Clean Tables

In [ ]:
import os
os.makedirs('../processed', exist_ok=True)

customers.to_csv('../processed/customers.csv', index=False)
products.to_csv('../processed/products.csv', index=False)
transactions.to_csv('../processed/transactions.csv', index=False)

# Orders table
orders = transactions.groupby(['invoice_id','customer_id','invoice_date']).agg(
    total_amount=('amount','sum'),
    num_items=('quantity','sum')
).reset_index()
orders.to_csv('../processed/orders.csv', index=False)

print(f"✅ Saved: customers({len(customers)}), transactions({len(transactions)}), orders({len(orders)})")
